In [1]:
# ============================================================
# INSTALL & IMPORTS
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

from scipy.stats import f_oneway, ttest_rel

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ============================================================
# CREATE OUTPUT FOLDER
# ============================================================
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("leaf_data.csv")

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# ============================================================
# FEATURE SELECTION
# ============================================================
feature_sets = {
    "All": "passthrough",
    "RFE": RFE(DecisionTreeClassifier(max_depth=5, random_state=42))
}

# ============================================================
# MODELS
# ============================================================
models = {
    "MLP": (
        MLPClassifier(
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=42
        ),
        True
    ),
    "RF": (
        RandomForestClassifier(random_state=42),
        False
    ),
    "KNN": (
        KNeighborsClassifier(),
        True
    )
}

# ============================================================
# PARAM GRID
# ============================================================
def get_param_grid(model_name, fs_name):
    param_grid = {}

    if fs_name == "RFE":
        param_grid["fs__n_features_to_select"] = [5, 6, 8]

    if model_name == "MLP":
        param_grid.update({
            "model__hidden_layer_sizes": [(30,), (50,), (30, 30)],
            "model__alpha": np.logspace(-3, 1, 5),
            "model__learning_rate_init": [0.0005, 0.001],
            "model__learning_rate": ["adaptive"]
        })

    elif model_name == "RF":
        param_grid.update({
            "model__n_estimators": [200, 500],
            "model__max_depth": [5, 10, 15],
            "model__min_samples_split": [5, 10],
            "model__min_samples_leaf": [2, 4],
            "model__max_features": ["sqrt"]
        })

    elif model_name == "KNN":
        param_grid.update({
            "model__n_neighbors": [5, 7, 9, 11],
            "model__weights": ["distance"],
            "model__p": [1, 2]
        })

    return param_grid

# ============================================================
# PIPELINE
# ============================================================
def build_pipeline(fs, model, scale):
    steps = [("fs", fs)]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

# ============================================================
# CV SETUP
# ============================================================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = {}

# ============================================================
# TRAIN LOOP
# ============================================================
for model_name, (model, scale) in models.items():
    for fs_name, fs in feature_sets.items():

        pipe = build_pipeline(fs, model, scale)
        param_grid = get_param_grid(model_name, fs_name)

        grid = GridSearchCV(
            pipe,
            param_grid,
            cv=inner_cv,
            scoring="f1_weighted",
            n_jobs=-1
        )

        scores = cross_val_score(
            grid,
            X, y,
            cv=outer_cv,
            scoring="f1_weighted"
        )

        results[f"{model_name}-{fs_name}"] = scores

# ============================================================
# SAVE RAW SCORES
# ============================================================
scores_df = pd.DataFrame(results)
scores_df.to_csv(os.path.join(output_dir, "cv_scores.csv"), index=False)

# ============================================================
# SUMMARY TABLE
# ============================================================
summary = []

for key, scores in results.items():
    mean = np.mean(scores)
    std = np.std(scores)
    ci = 1.96 * std / np.sqrt(len(scores))
    summary.append([key, mean, std, ci])

results_df = pd.DataFrame(summary, columns=["Model", "Mean", "Std", "CI"])
results_df = results_df.sort_values("Mean", ascending=False)

# SAVE TABLE
results_df.to_csv(os.path.join(output_dir, "results_summary.csv"), index=False)

# ============================================================
# STAT TESTS (SAVE TO FILE)
# ============================================================
with open(os.path.join(output_dir, "stat_tests.txt"), "w") as f:

    f.write("=== ANOVA TEST ===\n")
    f_stat, p_val = f_oneway(*results.values())
    f.write(f"F-stat: {f_stat}\n")
    f.write(f"p-value: {p_val}\n\n")

    f.write("=== PAIRED T-TEST ===\n")
    best_key = results_df.iloc[0]["Model"]
    best_scores = results[best_key]

    for key, scores in results.items():
        if key != best_key:
            t_stat, p = ttest_rel(best_scores, scores)
            f.write(f"{best_key} vs {key} -> p={p:.4f}\n")

# ============================================================
# BAR PLOT
# ============================================================
plt.figure(figsize=(10, 5))
plt.bar(results_df["Model"], results_df["Mean"], yerr=results_df["CI"], capsize=5)
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(os.path.join(output_dir, "bar_plot.png"))
plt.close()

# ============================================================
# BOX PLOT
# ============================================================
plt.figure(figsize=(10, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(os.path.join(output_dir, "box_plot.png"))
plt.close()

# ============================================================
# CONFUSION MATRIX
# ============================================================
best_key = results_df.iloc[0]["Model"]
best_model_name, best_fs_name = best_key.split("-")

model, scale = models[best_model_name]
fs = feature_sets[best_fs_name]

best_pipe = build_pipeline(fs, model, scale)
param_grid = get_param_grid(best_model_name, best_fs_name)

grid = GridSearchCV(best_pipe, param_grid, cv=3, scoring="f1_weighted", n_jobs=-1)

y_pred = cross_val_predict(grid, X, y, cv=outer_cv)

cm = confusion_matrix(y, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title(f"Confusion Matrix - {best_key}")
plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
plt.close()

print(f"\nAll outputs saved in: {output_dir}/")

C:\Users\Tech\AppData\Local\Temp\ipykernel_7668\674340002.py:206: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(results.values(), labels=results.keys())



All outputs saved in: outputs/
